# M4 · Model families

_AFP-AI · Domain 0 · ML Foundations_

**Compare a linear model with a tree ensemble on tabular ads-style data.**

We create nonlinear tabular signal, then compare logistic regression and a gradient boosting model. _Save a copy to your Drive (File -> Save a copy in Drive) to keep your edits._

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(4)

## Three families in formulas

Linear models use $f(x)=w^\top x+b$. Boosted trees add small trees, $F_M(x)=\sum_{m=1}^M \eta h_m(x)$. Neural nets compose layers, $f(x)=W_L\sigma(\cdots\sigma(W_1x))$.

In [ ]:
n = 4000
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
x3 = rng.normal(size=n)
interaction = (x1 > 0.4) & (x2 < -0.2)
logit = -2.2 + 0.4 * x3 + 2.2 * interaction.astype(float)
p = 1.0 / (1.0 + np.exp(-logit))
y = (rng.random(n) < p).astype(int)
X = np.column_stack([x1, x2, x3])

print("positive rate", round(y.mean(), 3))

## Step 1 - Split once

Both families get the same train/validation split so the comparison is fair.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=4, stratify=y)

assert abs(y_train.mean() - y_val.mean()) < 0.02

## Step 2 - Fit a linear baseline

The signal contains a threshold interaction, which a plain linear model cannot express directly.

In [ ]:
linear = LogisticRegression(max_iter=1000)
linear.fit(X_train, y_train)
p_linear = linear.predict_proba(X_val)[:, 1]
auc_linear = roc_auc_score(y_val, p_linear)

print("linear AUC", round(auc_linear, 3))

## Step 3 - Fit a boosted-tree model

The tree ensemble can split on $x_1>0.4$ and $x_2<-0.2$, then combine those regions.

In [ ]:
gbdt = GradientBoostingClassifier(random_state=4, n_estimators=80, max_depth=2, learning_rate=0.08)
gbdt.fit(X_train, y_train)
p_gbdt = gbdt.predict_proba(X_val)[:, 1]
auc_gbdt = roc_auc_score(y_val, p_gbdt)

print("GBDT AUC", round(auc_gbdt, 3))

assert auc_gbdt > auc_linear + 0.03

## Visualize the family comparison

This is not a universal law; it is a demonstration of matching family bias to tabular interactions.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["linear", "GBDT"], [auc_linear, auc_gbdt], color=["#4c78a8", "#54a24b"])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("validation AUC")
ax.set_title("model family matters")
plt.show()

## Practice

1. Add an explicit interaction feature `(x1 > 0.4) * (x2 < -0.2)` to the linear model.
2. Reduce `max_depth` to 1 and compare AUC.
3. Increase `n_estimators` and watch for overfitting.

In [ ]:
# Your turn:
